# Phase 1: Setup LIBERO + Download Demo Data

This notebook sets up the LIBERO benchmark environment and downloads human demonstration data.

**What we do here:**
1. Install system dependencies for headless MuJoCo rendering
2. Install LIBERO and its dependencies
3. Download demonstration data from HuggingFace
4. Inspect and visualize the demos
5. Verify environments run correctly

**Runtime:** GPU (T4 or better) — needed for EGL rendering

**Time:** ~15 min for setup, ~10 min per suite download

## 1. System Dependencies + EGL Setup

In [1]:
%%bash
# System dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev ffmpeg patchelf > /dev/null 2>&1

# NVIDIA EGL ICD config (missing on Colab by default)
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF
echo "System deps installed"

System deps installed


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Install LIBERO + Pin numpy

**After this cell runs, the runtime will restart. Skip to Cell 3 after reconnect.**

In [2]:
%%bash
# LIBERO-compatible robosuite + dependencies
pip install -q robosuite==1.4.1 robomimic==0.2.0 bddl==1.0.1
pip install -q hydra-core easydict einops cloudpickle "gym==0.25.2"
pip install -q imageio[ffmpeg] matplotlib h5py pandas Pillow tqdm rich
pip install -q huggingface-hub datasets

# Clone and install LIBERO
if [ ! -d "LIBERO" ]; then
    git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
fi
pip install -q -e LIBERO/

# Clone LIBERO-PRO for Phase 3
if [ ! -d "LIBERO-PRO" ]; then
    git clone --depth 1 https://github.com/Zxy-MLlab/LIBERO-PRO.git
fi

# Pin numpy: >=2.0 (Colab packages) and <2.1 (numba compat)
pip install -q "numpy>=2.0,<2.1"
echo "All packages installed"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.9/192.9 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.5/217.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.1 MB/s eta 0:00:00
All packages installed


Cloning into 'LIBERO'...
Updating files: 100% (1116/1116), done.
Cloning into 'LIBERO-PRO'...
Updating files: 100% (2288/2288), done.


In [ ]:
# Restart runtime to clear stale numpy bindings
# After restart, SKIP to Cell 3 below
import os
os.kill(os.getpid(), 9)

## 3. Environment Setup (run this after restart)

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

# Fix macros_private warning
import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated\n")

import numpy as np
print(f"numpy: {np.__version__}")
print(f"robosuite: {robosuite.__version__}")

from libero.libero import benchmark
print(f"LIBERO suites: {list(benchmark.get_benchmark_dict().keys())}")
print("Ready!")

## 4. Download Demonstration Data

LIBERO provides 50 human teleoperation demos per task, collected via SpaceMouse at 20Hz.

We download from HuggingFace Hub (faster than the original LIBERO script).

In [ ]:
from huggingface_hub import snapshot_download

# Download LIBERO-Spatial (start with the simplest suite)
# Other options: libero_object_no_noops, libero_goal_no_noops, libero_10_no_noops
SUITE = "libero_spatial"
REPO_ID = f"yifengzhu-hf/{SUITE}_no_noops"

local_dir = f"data/{SUITE}"
print(f"Downloading {REPO_ID}...")
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=local_dir,
)
print(f"Downloaded to {local_dir}")

# Show what we got
!find {local_dir} -type f | head -20

## 5. Explore Task Descriptions

List all tasks in LIBERO-Spatial with their natural language descriptions.

In [ ]:
from libero.libero import benchmark, get_libero_path

benchmark_dict = benchmark.get_benchmark_dict()

for suite_name in ["libero_spatial", "libero_object", "libero_goal"]:
    suite_obj = benchmark_dict[suite_name]()
    print(f"\n{'='*60}")
    print(f"  {suite_name} ({suite_obj.n_tasks} tasks)")
    print(f"{'='*60}")
    for i in range(suite_obj.n_tasks):
        task = suite_obj.get_task(i)
        print(f"  [{i}] {task.language}")

## 6. Inspect Demo Data

Look at the structure of the downloaded demo data.

In [ ]:
import h5py
from pathlib import Path

data_dir = Path(f"data/{SUITE}")

# Find data files
hdf5_files = sorted(data_dir.rglob("*.hdf5"))
parquet_files = sorted(data_dir.rglob("*.parquet"))
mp4_files = sorted(data_dir.rglob("*.mp4"))

print(f"HDF5 files: {len(hdf5_files)}")
print(f"Parquet files: {len(parquet_files)}")
print(f"MP4 files: {len(mp4_files)}")

# Inspect first HDF5 file if available
if hdf5_files:
    with h5py.File(hdf5_files[0], "r") as f:
        print(f"\nFile: {hdf5_files[0].name}")
        def show_tree(g, prefix=""):
            for k in sorted(g.keys()):
                item = g[k]
                if hasattr(item, "shape"):
                    print(f"{prefix}{k}: {item.shape} {item.dtype}")
                else:
                    print(f"{prefix}{k}/")
                    if len(list(item.keys())) <= 20:
                        show_tree(item, prefix + "  ")
                    else:
                        print(f"{prefix}  ({len(list(item.keys()))} items)")
        show_tree(f)

# Inspect LeRobot format if available
if parquet_files:
    import pandas as pd
    df = pd.read_parquet(parquet_files[0])
    print(f"\nParquet: {parquet_files[0].name}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Rows: {len(df)}")
    print(df.head())

## 7. Visualize Demo Episodes

Watch what the human demonstrations look like.

In [ ]:
import imageio
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import base64
from PIL import Image as PILImage

def show_gif(gif_path):
    with open(gif_path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(f'<img src="data:image/gif;base64,{data}" width="256">'))

# If we have HDF5 with image data
if hdf5_files:
    with h5py.File(hdf5_files[0], "r") as f:
        if "data" in f:
            demo_keys = sorted(f["data"].keys())
            demo = f["data"][demo_keys[0]]

            # Find camera images
            cam_key = None
            for k in ["agentview_image", "agentview_rgb"]:
                if "obs" in demo and k in demo["obs"]:
                    cam_key = k
                    break

            if cam_key:
                images = demo["obs"][cam_key][:]
                actions = demo["actions"][:]

                print(f"Demo: {demo_keys[0]}")
                print(f"  Steps: {len(actions)} ({len(actions)/20:.1f}s at 20Hz)")
                print(f"  Images: {images.shape}")

                # Save as GIF
                frames = []
                for i in range(0, len(images), 3):  # Every 3rd frame
                    frame = np.flip(images[i], axis=0)
                    frames.append(PILImage.fromarray(frame).resize((256, 256)))

                gif_path = "demo_preview.gif"
                frames[0].save(gif_path, save_all=True, append_images=frames[1:],
                              duration=50, loop=0, optimize=True)
                show_gif(gif_path)

                # Plot action trajectory
                fig, axes = plt.subplots(2, 1, figsize=(12, 6))
                labels = ["dx", "dy", "dz", "dax", "day", "daz", "gripper"]
                for i in range(3):
                    axes[0].plot(actions[:, i], label=labels[i])
                axes[0].legend()
                axes[0].set_title("Position Deltas")

                axes[1].plot(actions[:, -1], label="gripper", color="red")
                axes[1].legend()
                axes[1].set_title("Gripper State (-1=open, +1=close)")
                axes[1].set_xlabel("Step")

                plt.tight_layout()
                plt.show()
            else:
                print("No camera images in demo")

elif mp4_files:
    print(f"Data is in LeRobot format (MP4 videos). First video:")
    from IPython.display import Video
    display(Video(str(mp4_files[0]), embed=True, width=256))

## 8. Verify Environment Runs

Run a quick smoke test on a LIBERO task to confirm everything works.

In [ ]:
from libero.libero.envs import OffScreenRenderEnv

suite_obj = benchmark_dict["libero_spatial"]()
task = suite_obj.get_task(0)

bddl_file = os.path.join(
    get_libero_path("bddl_files"),
    task.problem_folder,
    task.bddl_file,
)

print(f"Task: {task.language}")
print(f"BDDL: {bddl_file}")

env = OffScreenRenderEnv(
    bddl_file_name=bddl_file,
    camera_heights=128,
    camera_widths=128,
)
env.seed(42)
obs = env.reset()

print(f"\nObservation keys: {list(obs.keys())}")
print(f"Action dim: {env.action_dim if hasattr(env, 'action_dim') else 7}")

# Run 50 random steps
frames = []
for i in range(50):
    action = np.random.uniform(-0.3, 0.3, size=7)
    obs, reward, done, info = env.step(action)
    if "agentview_image" in obs:
        frames.append(np.flip(obs["agentview_image"], axis=0))

# Show first and last frame
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(frames[0])
ax1.set_title("Step 0")
ax1.axis("off")
ax2.imshow(frames[-1])
ax2.set_title(f"Step {len(frames)}")
ax2.axis("off")
plt.suptitle(f"{task.language}", fontsize=11)
plt.tight_layout()
plt.show()

env.close()
print("\nEnvironment verification PASSED")

## 9. Save to Google Drive (Optional)

Persist data and repo clones across sessions.

In [ ]:
# Uncomment to save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r data/ /content/drive/MyDrive/AUTOLAB/libero_data/
# print("Saved to Google Drive")

## Phase 1 Complete!

**What we set up:**
- LIBERO + LIBERO-PRO repositories cloned
- Human demonstration data downloaded from HuggingFace
- Environments verified working with headless rendering

**Next:** Open `02_train_smolvla.ipynb` to fine-tune SmolVLA on this data.